# Problem: Compare Pretrained GloVe Embeddings vs Random Embeddings for Text Classification

### Problem Statement
You are tasked with implementing a **text classification model** on the **AG News** dataset using PyTorch. You will build two versions of the same classifier:
1. One using **pretrained GloVe embeddings** from TorchText
2. One using **randomly initialized embeddings**

Compare the performance of both approaches to understand the impact of transfer learning through pretrained word embeddings.

### Requirements
1. **Data Acquisition**: Load the AG News dataset (4-class news classification)
2. **Data Preprocessing**: Build vocabulary, tokenize text, and prepare data loaders
3. **Model Implementation**: 
   - Create a text classifier using an embedding layer + LSTM/GRU + fully connected layer
   - Implement two versions: one with GloVe embeddings, one with random embeddings
4. **Training**: Train both models with the same hyperparameters
5. **Evaluation**: Compare test accuracy and convergence speed

### Constraints
- Use GloVe embeddings (e.g., glove.6B.100d)
- Keep model architecture identical except for embedding initialization
- Use the same optimizer, learning rate, and training epochs for fair comparison
- AG News has 4 classes: World, Sports, Business, Sci/Tech

## Step 1: Data Acquisition
Load the AG News dataset using torchtext.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import torchtext
from torchtext.datasets import AG_NEWS
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import GloVe, build_vocab_from_iterator
from collections import Counter
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load AG News dataset
train_iter = AG_NEWS(split='train')
test_iter = AG_NEWS(split='test')

# Convert iterators to lists for easier manipulation
train_data = list(train_iter)
test_data = list(test_iter)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Sample: Label={train_data[0][0]}, Text={train_data[0][1][:100]}...")

## Step 2: Data Wrangling
Tokenize text, build vocabulary, and create data loaders.

In [ ]:
# Initialize tokenizer
tokenizer = get_tokenizer('basic_english')

# Build vocabulary from training data
def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens(train_data), specials=['<unk>', '<pad>'], min_freq=5)
vocab.set_default_index(vocab['<unk>'])

print(f"Vocabulary size: {len(vocab)}")

# Load GloVe embeddings
glove = GloVe(name='6B', dim=100)
print(f"GloVe vectors loaded: {len(glove.vectors)} words")

# Create embedding matrix from GloVe
embedding_dim = 100
vocab_size = len(vocab)
embedding_matrix = torch.zeros(vocab_size, embedding_dim)

words_found = 0
for i, word in enumerate(vocab.get_itos()):
    if word in glove.stoi:
        embedding_matrix[i] = glove.vectors[glove.stoi[word]]
        words_found += 1
    else:
        # Initialize with small random values for words not in GloVe
        embedding_matrix[i] = torch.randn(embedding_dim) * 0.01

print(f"Words found in GloVe: {words_found}/{vocab_size} ({100*words_found/vocab_size:.2f}%)")

In [ ]:
# Create custom dataset class
class TextDataset(Dataset):
    def __init__(self, data, vocab, tokenizer, max_length=200):
        self.data = data
        self.vocab = vocab
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        label, text = self.data[idx]
        # Tokenize and convert to indices
        tokens = self.tokenizer(text)
        # Truncate if too long
        tokens = tokens[:self.max_length]
        indices = [self.vocab[token] for token in tokens]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label - 1, dtype=torch.long)

# Collate function for padding
def collate_batch(batch):
    texts, labels = zip(*batch)
    # Pad sequences
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=vocab['<pad>'])
    labels = torch.stack(labels)
    return texts_padded, labels

# Create datasets
train_dataset = TextDataset(train_data, vocab, tokenizer)
test_dataset = TextDataset(test_data, vocab, tokenizer)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Step 3: Model Implementation
Define the text classifier architecture with embedding layer.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, 
                 n_layers=2, dropout=0.3, pretrained_embeddings=None, freeze_embeddings=False):
        super(TextClassifier, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab['<pad>'])
        
        # If pretrained embeddings provided, use them
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)
            if freeze_embeddings:
                self.embedding.weight.requires_grad = False
        
        # LSTM layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers, 
                           batch_first=True, dropout=dropout if n_layers > 1 else 0, bidirectional=True)
        
        # Fully connected layers
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch_size, seq_len, embedding_dim)
        
        # LSTM
        output, (hidden, cell) = self.lstm(embedded)
        # output shape: (batch_size, seq_len, hidden_dim*2)
        # hidden shape: (n_layers*2, batch_size, hidden_dim)
        
        # Concatenate the final forward and backward hidden states
        hidden_fwd = hidden[-2, :, :]
        hidden_bwd = hidden[-1, :, :]
        hidden_concat = torch.cat((hidden_fwd, hidden_bwd), dim=1)
        
        # Fully connected layer
        out = self.fc(self.dropout(hidden_concat))
        return out

# Model hyperparameters
hidden_dim = 128
output_dim = 4  # 4 classes in AG News
n_layers = 2
dropout = 0.3

# Initialize model with GloVe embeddings
model_glove = TextClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    dropout=dropout,
    pretrained_embeddings=embedding_matrix,
    freeze_embeddings=False
)

# Initialize model with random embeddings
model_random = TextClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    dropout=dropout,
    pretrained_embeddings=None,
    freeze_embeddings=False
)

print(f"Model with GloVe embeddings parameters: {sum(p.numel() for p in model_glove.parameters())}")
print(f"Model with random embeddings parameters: {sum(p.numel() for p in model_random.parameters())}")

## Step 4: Training
Train both models with identical hyperparameters.

In [ ]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=5):
    model.to(device)
    model.train()
    
    epoch_losses = []
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_idx, (texts, labels) in enumerate(train_loader):
            texts, labels = texts.to(device), labels.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Statistics
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (batch_idx + 1) % 200 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], '
                      f'Loss: {loss.item():.4f}, Acc: {100*correct/total:.2f}%')
        
        avg_loss = total_loss / len(train_loader)
        accuracy = 100 * correct / total
        epoch_losses.append(avg_loss)
        print(f'Epoch [{epoch+1}/{epochs}] - Avg Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')
    
    return epoch_losses

# Training configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()
learning_rate = 0.001
epochs = 5

print(f"Using device: {device}")
print("="*80)

In [ ]:
# Train model with GloVe embeddings
print("\n" + "="*80)
print("Training model with GloVe embeddings...")
print("="*80)

optimizer_glove = optim.Adam(model_glove.parameters(), lr=learning_rate)
losses_glove = train_model(model_glove, train_loader, criterion, optimizer_glove, device, epochs)

In [ ]:
# Train model with random embeddings
print("\n" + "="*80)
print("Training model with random embeddings...")
print("="*80)

optimizer_random = optim.Adam(model_random.parameters(), lr=learning_rate)
losses_random = train_model(model_random, train_loader, criterion, optimizer_random, device, epochs)

## Step 5: Evaluation
Evaluate and compare both models on the test set.

In [ ]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for texts, labels in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

# Evaluate both models
print("\n" + "="*80)
print("EVALUATION RESULTS")
print("="*80)

accuracy_glove = evaluate_model(model_glove, test_loader, device)
print(f"Test Accuracy (GloVe embeddings): {accuracy_glove:.2f}%")

accuracy_random = evaluate_model(model_random, test_loader, device)
print(f"Test Accuracy (Random embeddings): {accuracy_random:.2f}%")

print("\n" + "="*80)
print("COMPARISON SUMMARY")
print("="*80)
print(f"Accuracy improvement with GloVe: {accuracy_glove - accuracy_random:+.2f}%")
print(f"Relative improvement: {100*(accuracy_glove - accuracy_random)/accuracy_random:.2f}%")
print(f"\nFinal training loss (GloVe): {losses_glove[-1]:.4f}")
print(f"Final training loss (Random): {losses_random[-1]:.4f}")

print("\nKey Observations:")
print("1. Pretrained GloVe embeddings typically converge faster due to semantic initialization")
print("2. GloVe embeddings often achieve better test accuracy, especially with limited training data")
print("3. Random embeddings may require more epochs or data to match pretrained performance")
print("4. The gap narrows with more training data and longer training time")